# Convolutional Neural Networks — CIFAR-10

This notebook implements and evaluates a convolutional neural network using PyTorch and CIFAR-10.

The emphasis is on understanding the complete CNN workflow:

1. Load and preprocess image data.
2. Construct the CNN.
3. Train the model.
4. Evaluate generalization.
5. Analyze errors.
6. Run controlled experiments.


## 1. Imports

The notebook uses PyTorch for model construction and training, torchvision for CIFAR-10 and image transformations, and Matplotlib for visual inspection.


In [ ]:
import torch
import torch.nn as nn
from torch.optim import Adam
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import matplotlib.pyplot as plt


## 2. Device Selection

The training environment uses Intel's PyTorch XPU backend when available. Otherwise, training falls back to the CPU.


In [ ]:
if torch.xpu.is_available():
    device = torch.device("xpu")
else:
    device = torch.device("cpu")

print(f"Using device: {device}")


## 3. Data Preprocessing

Training images use random horizontal flips and random crops.

The test set is deliberately not augmented so that evaluation is performed on a consistent dataset.


In [ ]:
train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomCrop(32, padding=4),
    transforms.ToTensor(),
])

test_transform = transforms.Compose([
    transforms.ToTensor(),
])


## 4. Load CIFAR-10

CIFAR-10 contains 50,000 training images and 10,000 test images.

Each image is an RGB image with spatial dimensions of `32 × 32`.


In [ ]:
DATA_ROOT = "/data/Datasets/Images/First_CNN_CIFAR10_Dataset/"

train_dataset = datasets.CIFAR10(
    root=DATA_ROOT,
    train=True,
    download=True,
    transform=train_transform,
)

test_dataset = datasets.CIFAR10(
    root=DATA_ROOT,
    train=False,
    download=True,
    transform=test_transform,
)

print(f"Training samples: {len(train_dataset):,}")
print(f"Test samples: {len(test_dataset):,}")


## 5. DataLoaders

A batch size of 64 is used.

The training loader shuffles the training set. The test loader does not need to shuffle the test set.


In [ ]:
BATCH_SIZE = 64

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
)

images, labels = next(iter(train_loader))

print(f"Images: {images.shape}")
print(f"Labels: {labels.shape}")


## 6. CNN Architecture

The final architecture contains two convolutional blocks followed by a fully connected classifier.

```text
3 × 32 × 32
    │
    ▼
Conv2d(3 → 32)
    │
    ▼
ReLU
    │
    ▼
MaxPool(2 × 2)
    │
    ▼
Conv2d(32 → 64)
    │
    ▼
ReLU
    │
    ▼
MaxPool(2 × 2)
    │
    ▼
64 × 8 × 8
    │
    ▼
Flatten → 4096
    │
    ▼
Linear(4096 → 10)
```

The two pooling layers reduce:

```text
32 × 32 → 16 × 16 → 8 × 8
```

The second convolution produces 64 feature maps, so:

```text
64 × 8 × 8 = 4096
```

features enter the final classifier.


In [ ]:
class CNN(nn.Module):
    """Two-block convolutional neural network for CIFAR-10."""

    def __init__(self):
        super().__init__()

        self.conv1 = nn.Conv2d(
            in_channels=3,
            out_channels=32,
            kernel_size=3,
            stride=1,
            padding=1,
        )

        self.conv2 = nn.Conv2d(
            in_channels=32,
            out_channels=64,
            kernel_size=3,
            stride=1,
            padding=1,
        )

        self.relu = nn.ReLU()
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)
        self.flatten = nn.Flatten()

        self.fc = nn.Linear(
            in_features=64 * 8 * 8,
            out_features=10,
        )

    def forward(self, x):
        x = self.conv1(x)
        x = self.relu(x)
        x = self.pool(x)

        x = self.conv2(x)
        x = self.relu(x)
        x = self.pool(x)

        x = self.flatten(x)
        x = self.fc(x)

        return x


## 7. Initialize the Model

The model is moved to the selected accelerator before training.


In [ ]:
model = CNN().to(device)

print(model)


## 8. Loss Function and Optimizer

This is a 10-class classification problem.

`CrossEntropyLoss` is used directly with the model's raw logits. A separate softmax operation is not required before calculating the training loss.

Adam is used with a learning rate of `0.001`.


In [ ]:
loss_fn = nn.CrossEntropyLoss()

optimizer = Adam(
    model.parameters(),
    lr=0.001,
)


## 9. Training Loop

A complete epoch processes every batch in `train_loader`.

For each batch:

```text
Zero gradients
      ↓
Forward pass
      ↓
Calculate loss
      ↓
Backpropagation
      ↓
Update parameters
```

The loop records average loss and training accuracy.


In [ ]:
EPOCHS = 20

for epoch in range(EPOCHS):
    model.train()

    total_loss = 0.0
    correct = 0

    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)
        loss = loss_fn(outputs, labels)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        correct += (outputs.argmax(dim=1) == labels).sum().item()

    average_loss = total_loss / len(train_loader)
    accuracy = 100.0 * correct / len(train_dataset)

    print(
        f"Epoch: {epoch:02d} | "
        f"Loss: {average_loss:.8f} | "
        f"Correct: {correct:5d} | "
        f"Accuracy: {accuracy:.2f}%"
    )


## 10. Test Evaluation

`model.eval()` switches the model to evaluation mode.

`torch.no_grad()` disables gradient tracking because no parameter updates are being performed.


In [ ]:
model.eval()

correct = 0
predictions = []
actuals = []

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)
        predicted = outputs.argmax(dim=1)

        correct += (predicted == labels).sum().item()

        predictions.append(predicted.cpu())
        actuals.append(labels.cpu())

predictions = torch.cat(predictions)
actuals = torch.cat(actuals)

test_accuracy = 100.0 * correct / len(test_dataset)

print(f"Test accuracy: {test_accuracy:.2f}%")


## 11. Confusion Matrix

Rows represent actual classes and columns represent predicted classes.

- Diagonal entries are correct predictions.
- Off-diagonal entries are errors.

The matrix helps identify which classes the CNN tends to confuse.


In [ ]:
num_classes = 10

confusion = torch.zeros(
    num_classes,
    num_classes,
    dtype=torch.int64,
)

for actual, predicted in zip(actuals, predictions):
    confusion[actual, predicted] += 1

print(confusion)


## 12. Inspect Misclassified Images

Numerical metrics do not always explain why a prediction was wrong.

We can inspect individual failures and compare the actual and predicted classes.


In [ ]:
images, labels = next(iter(test_loader))

images = images.to(device)
labels = labels.to(device)

model.eval()

with torch.no_grad():
    outputs = model(images)
    predicted = outputs.argmax(dim=1)

wrong_indices = torch.where(predicted != labels)[0]

print(f"Misclassified images in batch: {len(wrong_indices)}")


In [ ]:
if len(wrong_indices) > 0:
    index = wrong_indices[0]

    plt.figure(figsize=(4, 4))
    plt.imshow(images[index].permute(1, 2, 0).cpu())
    plt.axis("off")
    plt.show()

    print(f"Actual:    {labels[index].item()}")
    print(f"Predicted: {predicted[index].item()}")
else:
    print("No misclassified images in this batch.")


# Experiments and Results

The experiments were designed to understand CNN behavior rather than exhaustively optimize CIFAR-10.

## Experiment 1 — Training Longer

| Experiment | Training Accuracy | Test Accuracy |
|---|---:|---:|
| 16 → 32, 10 epochs | 70.72% | 66.70% |
| 16 → 32, 20 epochs | 73.74% | 68.63% |

Training longer improved both training and test performance in this experiment.

## Experiment 2 — Increase Model Capacity

| Experiment | Training Accuracy | Test Accuracy |
|---|---:|---:|
| 32 → 64, 20 epochs | 81.92% | 70.27% |

The larger network fitted the training data substantially better, but the improvement on unseen data was smaller.

## Experiment 3 — Data Augmentation

| Experiment | Training Accuracy | Test Accuracy |
|---|---:|---:|
| 32 → 64 + horizontal flip | 76.98% | 72.24% |
| 32 → 64 + flip + random crop | 71.05% | **73.80%** |

Data augmentation reduced training accuracy while improving test accuracy.

## Key Takeaway

The central lesson is the difference between **fitting** and **generalization**.

A model can have lower training accuracy while achieving higher test accuracy when the training procedure encourages patterns that transfer better to unseen data.

---

## Next Topic

**Sequence Models → RNNs → LSTMs/GRUs → Attention → Transformers**
